
# 🧮 Week 2 — How LLMs Work: Probability, Sampling & Prompts

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/probability_lecture.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

📘 **Theme:** Understanding LLMs as probabilistic systems — linking mathematical intuition to model behavior and creative control.

---

### **Learning Objectives**
By the end of this week, you will be able to:
1. Write the probabilistic formulation of an LLM.
2. Explain how maximum likelihood training shapes model behavior.
3. Describe how temperature and entropy control randomness.
4. Analyze how prompts condition outputs.
5. Relate math intuition to real-world UX reliability.


In [1]:
# @title Setup (Run this first)
!git clone -q https://github.com/tulane-intro-ai-engineering/main.git
import sys; sys.path.append('/content/main')
from course_utils import lab2_setup

lab2_setup()
print("✅ Environment ready!")

Enter your OpenAI API key. It will only live in this Colab runtime.
OpenAI API key: ··········
✅ API key set.
✅ LAB2_colab_bootstrap complete — scientific libraries ready, helper function loaded.
✅ Environment ready!



## 🧩 Day 1 — A Simple Model of an LLM
---
**Guiding Question:**  
> What does it mean to say that an LLM “predicts the next token”?  
> How do probabilities create creativity *and* hallucinations?



### 🔢 The LLM as a Probability Model

A language model defines the probability of a sequence as:

$$
P(x_1, x_2, ..., x_T) = \prod_{t=1}^{T} P(x_t \mid x_{1:t-1})
$$

Each word is drawn from a conditional distribution based on what came before.


In [2]:
import numpy as np

tokens = ["is", "was", "will be", "seems"]
probs = np.array([0.65, 0.20, 0.10, 0.05])

for i in range(5):
    print(f"Sample {i+1}: {np.random.choice(tokens, p=probs)}")

Sample 1: is
Sample 2: seems
Sample 3: was
Sample 4: is
Sample 5: is



### 🧮 Maximum Likelihood Training

Training seeks parameters $\theta$ that make real text more probable:

$$
\theta^* = \arg\max_{\theta} \sum_{t=1}^{T} \log P_\theta(x_t \mid x_{1:t-1})
$$

Taking the log makes it easier to optimize (turns multiplication into addition).

In simple terms: the model rewards itself when it correctly predicts real text.


In [3]:
true_next = "mat"
pred_probs = {"mat": 0.7, "rug": 0.2, "dog": 0.1}

import numpy as np
print("True token:", true_next)
print("Predicted probabilities:", pred_probs)
print("Log-likelihood contribution:", np.log(pred_probs[true_next]))

True token: mat
Predicted probabilities: {'mat': 0.7, 'rug': 0.2, 'dog': 0.1}
Log-likelihood contribution: -0.35667494393873245



### 🎭 Why Likelihood ≠ Truth

The model optimizes *linguistic probability*, not *factual accuracy*:

$$
P(\text{"2 + 2 = 4"}) \approx 0.99, \quad P(\text{"2 + 2 = 5"}) \approx 0.01
$$

For unseen prompts, it predicts what **sounds** likely — even if false.

> **Hallucination:** high $P(\text{text}|\text{context})$, low $P(\text{truth}|\text{world})$.



### 🧩 Unifying Diagram v1 — Adding Training Data Distribution

![Unifying Diagram v1](sandbox:/mnt/data/A_flowchart_diagram_illustrates_the_architecture_o.png)

Everything the model knows comes from its **training data distribution**, which shapes its probabilities.


In [ ]:
# 🧠 Concept Check — Day 1
answer1 = "conditional probabilities"
answer2 = "maximize likelihood of real data"

assert "conditional" in answer1.lower()
assert "likelihood" in answer2.lower()
print("✅ Passed: You understand the core probabilistic model.")


## 💻 Day 2 — Sampling, Temperature, and Prompt Patterns
---
**Guiding Question:**  
> How does randomness influence creativity and determinism in LLMs?



### 🌡️ Temperature and Softmax Sampling

Models sample from a softmax distribution:

$$
P_T(x_t=i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}
$$

- **Low T (e.g., 0.3):** sharper distribution → repetitive, factual.  
- **High T (e.g., 2.0):** flatter distribution → creative, variable.


In [4]:
import numpy as np

def softmax_temp(z, T):
    p = np.exp(z / T)
    p /= p.sum()
    return p

logits = np.array([2.0, 1.0, 0.1])
for T in [0.3, 1.0, 2.0]:
    print(f"T={T}: {softmax_temp(logits, T)}")

T=0.3: [0.96390178 0.03438623 0.00171199]
T=1.0: [0.65900114 0.24243297 0.09856589]
T=2.0: [0.50168776 0.30428901 0.19402324]



### 🔀 Entropy: Measuring Uncertainty

Entropy quantifies unpredictability:

$$
H(P) = -\sum_i P_i \log_2 P_i
$$

- Low $H$ → confident, stable predictions  
- High $H$ → uncertain, diverse outputs


In [5]:
import math

def entropy(p): return -sum(pi * math.log(pi, 2) for pi in p)

print("Entropy([0.9, 0.1]) =", entropy([0.9, 0.1]))
print("Entropy([0.5, 0.5]) =", entropy([0.5, 0.5]))

Entropy([0.9, 0.1]) = 0.4689955935892812
Entropy([0.5, 0.5]) = 1.0



### 🧠 Prompt Patterns and Conditioning

Prompts provide *context* that reshapes token probabilities:

$$
P_\theta(x_t \mid x_{1:t-1}, \text{prompt})
$$

Common patterns include:
1. **Role prompts:** “You are a teacher explaining entropy.”  
2. **Example prompts:** “Q: ... A: ...”  
3. **Constraint prompts:** “Answer in three bullet points.”



### 🧪 Bridge to Lab 2 — Measuring Diversity

In Lab 2, you will vary `temperature` and measure diversity:

$$
\text{Diversity Index} = \frac{\text{unique tokens}}{\text{total tokens}}
$$

You’ll test how higher temperature increases diversity but may reduce accuracy.


In [ ]:
# 🧠 Concept Check — Day 2
answer1 = "controls randomness"
answer2 = "higher"

assert "random" in answer1.lower()
assert "high" in answer2.lower()
print("✅ Passed: You understand temperature and entropy!")


<details>
<summary>🧑‍🏫 Instructor Notes</summary>

**Day 1 Focus:**  
- Encourage intuition for $$P(x_t | x_{1:t-1})$$.  
- Visualize probabilities as "heat maps" or simple bar charts.  

**Day 2 Focus:**  
- Compare outputs at different temperatures live.  
- Discuss how prompt phrasing shifts distributions.  
- Connect entropy to creativity and reliability.

**Extensions:**  
- Add `top_p` sampling visualization.  
- Ask: “What happens if we combine low T and long prompts?”
</details>
